# Across-record defects: natural keys, dedup decisions, and the holdings contract

This notebook covers the second tier of the case study (§2.2): defects you cannot see by looking at one row at a time. Each row is individually valid; the problem shows up only when you compare rows to each other.

The brief describes three shapes of this problem:

- **A1**: the same holding is delivered twice in the same sync, under two different `investment_id`s.
- **A2**: the same holding disappears under one id and reappears later under a new one. The old id freezes at its last value, and if we do nothing, the portfolio double-counts.
- **A3**: a holding shows up valued at zero in one sync, then above zero in the next, with its quantity unchanged.

For each defect we have to decide three things: which rows to **reject** outright, which ones to **quarantine** (hold back into a separate table), and which ones to **admit with a warning**. And whichever we choose, a downstream consumer needs a way to tell what happened.

The notebook walks through all three defects against the built warehouse, ends with a decision matrix, and shows how it lands in dbt.

| § | question the section answers | how it answers it |
|---|---|---|
| 1 | what identifies one physical holding, since the provider's id is unreliable? | pick a small set of columns per family (the "natural key"), verified two ways |
| 2 | when a sync ships the same holding under two ids, what kind of duplicate is it? | classify into one of three shapes: identical copy, real split into lots, or same size with different values |
| 3 | for the "different values" case from §2, which value do we keep? | the non-zero one: the zero side is always a stale reading, and transactions confirm both ids are one holding |
| 4 | when the provider retires an id and uses a new one for the same holding, do we lose the position? | no: the natural key from §1 stitches the old id and the new one into one continuous holding |
| 5 | when a value drops to zero for one sync and bounces back, is that real? | no, if the quantity was unchanged around it: we keep the delivered zero but flag it, never smooth it away |
| 6 | what do we actually do with each defect? | the decision matrix, built from the live counts above |
| 7 | how does this land in dbt? | one `holdings_*` view per family, a quarantine relation for unresolvable cases, and `dq_flags` on every holding |

**Source tables.** `stg_openfinance__*_positions_detail`, `*_positions_balances`, `*_transactions`. Run `make build` first if the warehouse is empty.

**How to play with this notebook.** Everything is a small helper on top of one config dict (`FAMILIES`). Three helpers are worth remembering:
- `inspect(family, subclass, idx)`: show any duplicate group side by side, one column per `investment_id`.
- `tx_stream(family, ids)`: show the movement history of any set of ids.
- `key_timeline(family, key_row)`: show one holding across every sync.

Change the arguments and re-run to explore.


## 0 · Setup

One config dict, `FAMILIES`, drives every query in the notebook. For each family it declares:

- `key`: the columns that identify one physical holding (this is what §1 is about).
- `qty` / `gross`: the quantity and value columns in the balances table.
- `tx_qty` / `tx_date` / `tx_amount`: the corresponding columns in the transactions table.
- `moving_tx`: the set of `transaction_type` values that actually change the position (buys, sells, transfers). Cash-only events like dividends and interest are excluded.
- `rationale`: a short note explaining the key choice.


In [1]:
import os
import duckdb
import pandas as pd

pd.set_option('display.max_rows', 80)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 90)
pd.set_option('display.width', 220)

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

con = duckdb.connect('warehouse.duckdb', read_only=True)

FAMILIES = {
    'variable_incomes': {
        'key': ['ticker'],
        'qty': 'quantity', 'gross': 'gross_amount',
        'tx_qty': 'transaction_quantity', 'tx_date': 'transaction_date', 'tx_amount': 'transaction_amount',
        'moving_tx': {'COMPRA', 'VENDA', 'TRANSFERENCIA_TITULARIDADE'},
        'rationale': 'ticker alone: ISIN is blank in ~900 rows and would fragment the key; no account quotes two ISINs under one ticker (verified in 1.2), so ISIN is a redundant attribute here',
    },
    'funds': {
        'key': ['fund_cnpj'],
        'qty': 'quota_quantity', 'gross': 'gross_amount',
        'tx_qty': 'transaction_quota_quantity', 'tx_date': 'transaction_conversion_date', 'tx_amount': 'transaction_amount',
        'moving_tx': {'APLICACAO', 'RESGATE', 'TRANSFERENCIA_TITULARIDADE'},
        'rationale': 'one open-ended fund maps to exactly one CNPJ',
    },
    'bank_fixed_incomes': {
        'key': ['isin_code', 'issuer_cnpj', 'issue_date', 'due_date'],
        'qty': 'quantity', 'gross': 'gross_amount',
        'tx_qty': 'transaction_quantity', 'tx_date': 'transaction_date', 'tx_amount': 'transaction_gross_amount',
        'moving_tx': {'APLICACAO', 'RESGATE', 'VENCIMENTO', 'AMORTIZACAO',
                      'TRANSFERENCIA_CUSTODIA', 'TRANSFERENCIA_TITULARIDADE'},
        'rationale': 'banks re-use ISINs across CDB issuances; (issuer, issue_date, due_date) pins one issuance',
    },
    'credit_fixed_incomes': {
        'key': ['isin_code', 'debtor_cnpj', 'due_date'],
        'qty': 'quantity', 'gross': 'gross_amount',
        'tx_qty': 'transaction_quantity', 'tx_date': 'transaction_date', 'tx_amount': 'transaction_gross_amount',
        'moving_tx': {'APLICACAO', 'RESGATE', 'VENCIMENTO', 'AMORTIZACAO',
                      'TRANSFERENCIA_CUSTODIA', 'TRANSFERENCIA_TITULARIDADE'},
        'rationale': 'debtor and maturity are not enough (13 accounts hold two real ISINs under one (debtor, due) pair: verified in 1.2), so ISIN must stay in the key despite being blank in ~220 rows',
    },
    'treasure_titles': {
        'key': ['product_name'],
        'qty': 'quantity', 'gross': 'gross_amount',
        'tx_qty': 'transaction_quantity', 'tx_date': 'transaction_date', 'tx_amount': 'transaction_gross_amount',
        'moving_tx': {'APLICACAO', 'RESGATE', 'VENCIMENTO', 'AMORTIZACAO',
                      'TRANSFERENCIA_CUSTODIA', 'TRANSFERENCIA_TITULARIDADE'},
        'rationale': 'Tesouro Direto has one title per name ("Tesouro Selic 2031"); ISIN would be the cleaner key but is blank in ~400 rows, and no account maps one name to two real ISINs (verified in 1.2)',
    },
}

def detail(family): return f'stg_openfinance__{family}_positions_detail'
def balances(family): return f'stg_openfinance__{family}_positions_balances'
def transactions(family): return f'stg_openfinance__{family}_transactions'
def query(sql): return con.execute(sql).fetchdf()

FINDINGS = {}  # each section deposits its headline numbers; §6 assembles them into the decision matrix

for family in FAMILIES:
    n_rows = con.execute(f'SELECT count(*) FROM {detail(family)}').fetchone()[0]
    print(f'{family:22s} {n_rows:>7,} detail rows')


variable_incomes        16,479 detail rows
funds                    8,355 detail rows
bank_fixed_incomes      64,535 detail rows
credit_fixed_incomes     5,447 detail rows
treasure_titles          9,138 detail rows


## 1 · The natural key: what identifies one physical holding

The provider gives us `investment_id`, but that is *the provider's* label for a row, not the label of the physical holding underneath. The whole tier-2 problem exists because the two don't always match: the same real holding can arrive under two `investment_id`s in one sync, or under a different `investment_id` after a swap.

So we need our own identifier: a **natural key**. That is the set of columns in the `detail` table which, taken together, identify one physical holding **within one account**, no matter which id the provider stamped on it.

- Same natural key = same holding.
- Different natural key = different holding.

Every downstream operation (dedup, summing lots, stitching a holding across syncs) depends on this. So below we declare the key per family, and then run two checks to make sure it can carry that weight.


In [2]:
with pd.option_context('display.max_colwidth', None):
    display(pd.DataFrame([
        {'family': family, 'natural_key': ' + '.join(cfg['key']), 'rationale': cfg['rationale']}
        for family, cfg in FAMILIES.items()
    ]))


,family,natural_key,rationale
0,variable_incomes,ticker,"ticker alone: ISIN is blank in ~900 rows and would fragment the key; no account quotes two ISINs under one ticker (verified in 1.2), so ISIN is a redundant attribute here"
1,funds,fund_cnpj,one open-ended fund maps to exactly one CNPJ
2,bank_fixed_incomes,isin_code + issuer_cnpj + issue_date + due_date,"banks re-use ISINs across CDB issuances; (issuer, issue_date, due_date) pins one issuance"
3,credit_fixed_incomes,isin_code + debtor_cnpj + due_date,"debtor and maturity are not enough (13 accounts hold two real ISINs under one (debtor, due) pair: verified in 1.2), so ISIN must stay in the key despite being blank in ~220 rows"
4,treasure_titles,product_name,"Tesouro Direto has one title per name (""Tesouro Selic 2031""); ISIN would be the cleaner key but is blank in ~400 rows, and no account maps one name to two real ISINs (verified in 1.2)"


### 1.1 · Two trust gates

A natural key is only useful if we can trust it. Two things can go wrong:

1. **NULL in a key column.** A row with `NULL` in one of its key columns silently drops out of every `GROUP BY` below and never gets deduped. If any such rows exist, the key is not usable. (Missing identity fields are actually a within-record defect, handled in `02_within_record_defects.ipynb`. Here we just double-check that none survived into staging.)
2. **The key changes for the same `investment_id`.** If one `investment_id` has different key values in different snapshots, one of two things is true: the columns we picked are not really identity (they change over time), or the provider reused the same id for a different holding. Either way, the key cannot reliably stitch identity across syncs.

Both counts must be zero. The asserts below stop the notebook if a future rebuild breaks this.

There is also a third, sneakier failure mode: a key that looks fine to `IS NOT NULL` but **fragments** because some rows carry an empty string in a key column. That one shaped the actual key choices above; §1.2 hunts it.


In [3]:
rows = []
for family, cfg in FAMILIES.items():
    any_key_null = ' OR '.join(f'{col} IS NULL' for col in cfg['key'])
    key_tuple = ', '.join(cfg['key'])
    null_rows = con.execute(f'SELECT count(*) FROM {detail(family)} WHERE {any_key_null}').fetchone()[0]
    unstable = con.execute(f'''
        SELECT count(*) FROM (
            SELECT investment_id FROM {detail(family)}
            GROUP BY investment_id
            HAVING count(DISTINCT ({key_tuple})) > 1)''').fetchone()[0]
    rows.append({'family': family, 'null_key_rows': null_rows, 'ids_with_unstable_key': unstable})
gates = pd.DataFrame(rows)
print(gates.to_string(index=False))

assert gates['null_key_rows'].eq(0).all(), \
    'rows with a NULL key column would silently escape every grouping below: fix upstream first'
assert gates['ids_with_unstable_key'].eq(0).all(), \
    'an id that changes key across snapshots means the key is wrong or the id was recycled: investigate first'
print('\nboth gates pass: the key never leaks (no NULL keys) and never lies (no id changes key)')


              family  null_key_rows  ids_with_unstable_key
    variable_incomes              0                      0
               funds              0                      0
  bank_fixed_incomes              0                      0
credit_fixed_incomes              0                      0
     treasure_titles              0                      0

both gates pass: the key never leaks (no NULL keys) and never lies (no id changes key)


### 1.2 · Choosing keys by falsification: the blank-ISIN trap

Gate 1 checks `IS NOT NULL`, but an empty string `''` passes it. Three families ship **blank** ISINs
in the hundreds of rows, and a blank inside a key column is worse than a NULL: it silently fragments
the key ("`''` vs `BRPETRACNPR6`" reads as two different holdings) instead of failing loudly.

So the keys in §1 were not guessed. For each family we started from the obvious ISIN-based key and
tried to break the alternative:

**variable_incomes**: the obvious key is `(isin_code, ticker)`, but ~900 rows ship a blank ISIN, so
that key would split one holding into two ("blank PETR4" and "BRPETRACNPR6 PETR4"). Dropping ISIN is
only safe if `ticker` alone never collides, i.e. if no account ever quotes two *real* ISINs under one
ticker. The check finds **zero** such accounts, so `ticker` alone is the key and ISIN demotes to an
attribute.

**treasure_titles**: same story with `product_name`. ~400 rows have a blank ISIN, and no account maps
one product name ("Tesouro Selic 2031") to two real ISINs, so `product_name` is the key.

**credit_fixed_incomes**: here the escape hatch fails. Keying on `(debtor_cnpj, due_date)` without
ISIN *would* collide: **13 accounts** hold two different real ISINs under the same (debtor, due)
pair, so ISIN must stay in the key. The 224 rows whose ISIN is blank therefore cannot be keyed at
all: they are admitted at **lot grain, never merged with anything**, carrying the `missing_identity`
flag (§6).

The asserts below re-run these checks on every rebuild, so a data refresh that breaks a key choice
stops the notebook here.


In [4]:
# --- how many blanks per family? ('' passes IS NOT NULL: this is the trap)
audit_rows = []
for family in FAMILIES:
    cols = query(f"SELECT * FROM {detail(family)} LIMIT 0").columns
    if 'isin_code' not in cols:
        continue
    n_blank = con.execute(
        f"SELECT count(*) FROM {detail(family)} WHERE trim(coalesce(isin_code, '')) = ''").fetchone()[0]
    audit_rows.append({'family': family, 'blank_isin_rows': n_blank, 'isin_in_key': 'isin_code' in FAMILIES[family]['key']})
blank_audit = pd.DataFrame(audit_rows)
print(blank_audit.to_string(index=False))

# --- falsification 1: is ticker alone safe for variable_incomes?
tickers_with_two_real_isins = con.execute(f'''
    SELECT count(*) FROM (
        SELECT account_id, ticker FROM {detail('variable_incomes')}
        WHERE trim(coalesce(isin_code, '')) <> ''
        GROUP BY account_id, ticker
        HAVING count(DISTINCT isin_code) > 1)''').fetchone()[0]
table = detail('variable_incomes')
blank_ticker_rows = con.execute(
    f"SELECT count(*) FROM {table} WHERE trim(coalesce(ticker, '')) = ''").fetchone()[0]
assert tickers_with_two_real_isins == 0, 'an account quotes two real ISINs under one ticker: ticker alone is NOT a safe key'
assert blank_ticker_rows == 0, 'blank tickers found: ticker would fragment just like ISIN does'
print(f'\nvariable_incomes: 0 accounts with two real ISINs under one ticker, 0 blank tickers -> ticker is the key')

# --- falsification 2: is product_name alone safe for treasure_titles?
names_with_two_real_isins = con.execute(f'''
    SELECT count(*) FROM (
        SELECT account_id, product_name FROM {detail('treasure_titles')}
        WHERE trim(coalesce(isin_code, '')) <> ''
        GROUP BY account_id, product_name
        HAVING count(DISTINCT isin_code) > 1)''').fetchone()[0]
table = detail('treasure_titles')
blank_name_rows = con.execute(
    f"SELECT count(*) FROM {table} WHERE trim(coalesce(product_name, '')) = ''").fetchone()[0]
assert names_with_two_real_isins == 0, 'an account maps one product_name to two real ISINs: product_name is NOT a safe key'
assert blank_name_rows == 0, 'blank product_names found: the key would fragment'
print(f'treasure_titles:  0 accounts with two real ISINs under one product_name, 0 blank names -> product_name is the key')

# --- falsification 3: can credit_fixed drop isin and key on (debtor, due)?
debtor_due_collisions = con.execute(f'''
    SELECT count(*) FROM (
        SELECT account_id, debtor_cnpj, due_date FROM {detail('credit_fixed_incomes')}
        WHERE trim(coalesce(isin_code, '')) <> ''
        GROUP BY account_id, debtor_cnpj, due_date
        HAVING count(DISTINCT isin_code) > 1)''').fetchone()[0]
table = detail('credit_fixed_incomes')
unkeyable_credit_rows = con.execute(
    f"SELECT count(*) FROM {table} WHERE trim(coalesce(isin_code, '')) = ''").fetchone()[0]
assert debtor_due_collisions > 0, 'if this ever drops to 0, (debtor, due) becomes a viable blank-proof key: revisit'
print(f'credit_fixed:     {debtor_due_collisions} (account, debtor, due) pairs hold >1 real ISIN -> isin_code must stay in the key')
print(f'                  {unkeyable_credit_rows} rows with blank isin cannot be keyed -> admit at lot grain with dq_flags=[missing_identity]')
FINDINGS['missing_identity_rows'] = unkeyable_credit_rows

def key_ok(family, alias='d'):
    # every grouping below filters on this: NULL *or blank* key columns opt a row out of
    # identity resolution entirely: those rows travel alone as missing_identity (§6)
    prefix = f'{alias}.' if alias else ''
    return ' AND '.join(
        f'{prefix}{col} IS NOT NULL' if col.endswith('_date')
        else f"trim(coalesce({prefix}{col}, '')) <> ''"
        for col in FAMILIES[family]['key'])

print('\nkey_ok(family) is the reusable predicate: e.g. for credit_fixed:', key_ok('credit_fixed_incomes'))


              family  blank_isin_rows  isin_in_key
    variable_incomes              906        False
               funds             8355        False
  bank_fixed_incomes                0         True
credit_fixed_incomes              224         True
     treasure_titles              404        False

variable_incomes: 0 accounts with two real ISINs under one ticker, 0 blank tickers -> ticker is the key
treasure_titles:  0 accounts with two real ISINs under one product_name, 0 blank names -> product_name is the key
credit_fixed:     13 (account, debtor, due) pairs hold >1 real ISIN -> isin_code must stay in the key
                  224 rows with blank isin cannot be keyed -> admit at lot grain with dq_flags=[missing_identity]

key_ok(family) is the reusable predicate: e.g. for credit_fixed: trim(coalesce(d.isin_code, '')) <> '' AND trim(coalesce(d.debtor_cnpj, '')) <> '' AND d.due_date IS NOT NULL


## 2 · A1: the same holding twice in one sync

The A1 defect: the provider ships the same physical holding more than once in a single sync, under different `investment_id`s.

**How we detect it.** Group every row by `(snapshot_id, account_id, natural_key)`. If a group has more than one `investment_id`, we have an A1 duplicate.

**How we classify it.** Not every duplicate is the same problem. We compare the `quantity` and `gross_amount` values of the rows in each group and sort them into one of three cases:

| # | if... | subclass | what it means |
|---|---|---|---|
| 1 | quantities differ across the ids | `partition` | these are real lots of one holding, split across ids. To get the true total, we **sum** them. |
| 2 | quantities agree AND gross amounts agree | `hard_dup` | the rows are identical copies. Dropping all but one loses nothing. |
| 3 | quantities agree BUT gross amounts differ | `conflict` | same position size, two different valuations. Someone is wrong; §3 figures out who. |

The three cases are exhaustive: every duplicate group falls into exactly one, so nothing slips through unclassified.

**A note on NULLs.** `count(DISTINCT col)` in SQL ignores NULLs, so a group where every row has a NULL quantity reads as "all agree" and falls through to the gross-amount test. That is the behavior we want here: the group has no evidence to distinguish its members on quantity, so we let the next test decide.

The classifier below is one SQL string. You can copy it into a DuckDB shell to run it directly, and §7 uses it almost verbatim as the dbt detection test.


In [5]:
def classify_sql(family):
    cfg = FAMILIES[family]
    key_cols = ', '.join(f'd.{col}' for col in cfg['key'])
    usable_key = key_ok(family)
    return f'''SELECT
    d.snapshot_id, d.account_id, {key_cols},
    count(DISTINCT d.investment_id)                              AS n_ids,
    array_agg(DISTINCT d.investment_id ORDER BY d.investment_id) AS investment_ids,
    any_value(d.institution_name)                                AS institution_name,
    count(DISTINCT b.{cfg['qty']})                                 AS n_qty,
    count(DISTINCT b.{cfg['gross']})                               AS n_gross,
    count(*) FILTER (WHERE b.{cfg['gross']} = 0)                   AS n_zero_gross,
    min(b.{cfg['qty']})                                            AS qty_agreed,
    CASE WHEN count(DISTINCT b.{cfg['qty']})   > 1 THEN 'partition'
         WHEN count(DISTINCT b.{cfg['gross']}) <= 1 THEN 'hard_dup'
         ELSE 'conflict' END                                     AS subclass
FROM {detail(family)} d
LEFT JOIN {balances(family)} b USING (snapshot_id, investment_id)
WHERE {usable_key}
GROUP BY d.snapshot_id, d.account_id, {key_cols}
HAVING count(DISTINCT d.investment_id) > 1
ORDER BY n_ids DESC, d.snapshot_id, d.account_id, {key_cols}'''

def classify(family):
    return query(classify_sql(family))

print(classify_sql('bank_fixed_incomes'))


SELECT
    d.snapshot_id, d.account_id, d.isin_code, d.issuer_cnpj, d.issue_date, d.due_date,
    count(DISTINCT d.investment_id)                              AS n_ids,
    array_agg(DISTINCT d.investment_id ORDER BY d.investment_id) AS investment_ids,
    any_value(d.institution_name)                                AS institution_name,
    count(DISTINCT b.quantity)                                 AS n_qty,
    count(DISTINCT b.gross_amount)                               AS n_gross,
    count(*) FILTER (WHERE b.gross_amount = 0)                   AS n_zero_gross,
    min(b.quantity)                                            AS qty_agreed,
    CASE WHEN count(DISTINCT b.quantity)   > 1 THEN 'partition'
         WHEN count(DISTINCT b.gross_amount) <= 1 THEN 'hard_dup'
         ELSE 'conflict' END                                     AS subclass
FROM stg_openfinance__bank_fixed_incomes_positions_detail d
LEFT JOIN stg_openfinance__bank_fixed_incomes_positions_balances b USING (snapshot_i

### 2.1 · The census

How many duplicate groups exist per family and per subclass. The extra column `partition_with_zero_lot` counts partitions that contain at least one lot with `gross = 0`. Those sums will **understate** the holding's true value: if you sum three lots and one of them is priced at zero, the total is missing that lot's real value. This gets its own flag in the decision matrix (§6), and it is the same problem A3 catches at a different grain (§5).


In [6]:
frames = []
for family in FAMILIES:
    classified = classify(family)
    classified.insert(0, 'family', family)
    frames.append(classified)
dup_groups = pd.concat(frames, ignore_index=True)

census = (dup_groups.pivot_table(index='family', columns='subclass', values='n_ids', aggfunc='size', fill_value=0)
            .reindex(FAMILIES.keys())
            .reindex(columns=['hard_dup', 'partition', 'conflict'], fill_value=0))
census['total'] = census.sum(axis=1)
census['partition_with_zero_lot'] = [
    len(dup_groups.query("family == @family and subclass == 'partition' and n_zero_gross > 0")) for family in FAMILIES
]
print(census.to_string())

FINDINGS['hard_dup'] = int(census['hard_dup'].sum())
FINDINGS['partition'] = int(census['partition'].sum())
FINDINGS['conflict'] = int(census['conflict'].sum())
FINDINGS['partition_with_zero_lot'] = int(census['partition_with_zero_lot'].sum())


subclass              hard_dup  partition  conflict  total  partition_with_zero_lot
family                                                                             
variable_incomes             0       1729         0   1729                       22
funds                      115        464         0    579                       79
bank_fixed_incomes        1153         45        83   1281                        0
credit_fixed_incomes       160          7         0    167                        0
treasure_titles              0        730         0    730                        0


### 2.2 · Who ships what

Every staging row carries the `institution_name`, so we can attribute each duplicate group to the institution that produced it with one `GROUP BY`. This is the table you would bring to a provider call: `conflict` groups are concentrated in a handful of institutions, while `partition` groups appear across the ecosystem (they are normal behavior, not a bug).


In [7]:
attribution = (dup_groups.groupby(['subclass', 'family', 'institution_name']).size()
              .rename('groups').reset_index()
              .sort_values(['subclass', 'family', 'groups'], ascending=[True, True, False]))
print(attribution.to_string(index=False))


 subclass               family institution_name  groups
 conflict   bank_fixed_incomes             Itau      48
 conflict   bank_fixed_incomes    Banco XP S.A.      22
 conflict   bank_fixed_incomes          C6 Bank      13
 hard_dup   bank_fixed_incomes           Nubank     493
 hard_dup   bank_fixed_incomes             Itau     237
 hard_dup   bank_fixed_incomes      BTG Banking     175
 hard_dup   bank_fixed_incomes   Banco Inter PF      78
 hard_dup   bank_fixed_incomes    Banco XP S.A.      76
 hard_dup   bank_fixed_incomes          C6 Bank      67
 hard_dup   bank_fixed_incomes           PicPay      20
 hard_dup   bank_fixed_incomes  Banco do Brasil       7
 hard_dup credit_fixed_incomes           Nubank      63
 hard_dup credit_fixed_incomes             Itau      31
 hard_dup credit_fixed_incomes   Banco Inter PF      29
 hard_dup credit_fixed_incomes    Banco XP S.A.      20
 hard_dup credit_fixed_incomes          C6 Bank       9
 hard_dup credit_fixed_incomes      BTG Banking 

### 2.3 · The microscope: look at one group up close

The counts above are aggregates. Before trusting them, look at actual rows.

`inspect(family, subclass, idx)` pulls **every** column of both the `detail` and `balances` tables for one duplicate group, lays them out with one column per `investment_id`, and marks the columns where the ids disagree. Change `family`, `subclass`, or `idx` to explore different groups; pass `biggest=True` to see the most fragmented one.

**How to read the output.** The whole classification from §2 applied by eye, plus one extra safety check:

- Only quantities and money differ → real lots of one holding (`partition`).
- Quantities agree, gross amounts differ → valuation `conflict`.
- Every column agrees → redundant copy (`hard_dup`).
- The **identity** columns themselves differ (e.g. two different ISINs, two different issue dates) → the natural key is wrong. Fix the key, not the data.


In [8]:
def side_by_side(family, group_row):
    ids = list(group_row['investment_ids'])
    ids_sql = ', '.join(f"'{i}'" for i in ids)
    parts = []
    for label, table in (('detail', detail(family)), ('balances', balances(family))):
        block = query(f"SELECT * FROM {table} WHERE snapshot_id = '{group_row['snapshot_id']}'"
                      f" AND investment_id IN ({ids_sql})").set_index('investment_id').T
        block.index = f'{label}.' + block.index
        parts.append(block)
    both = pd.concat(parts)
    both.columns = [col[:8] for col in both.columns]
    both['differs'] = both.astype(str).nunique(axis=1) > 1
    key_desc = '  '.join(f'{col}={group_row[col]}' for col in FAMILIES[family]['key'])
    print(f"[{family}] subclass={group_row['subclass']}  snapshot={group_row['snapshot_id'][:8]}  "
          f"account={group_row['account_id'][:8]}  n_ids={group_row['n_ids']}")
    print(f'  key: {key_desc}\n')
    return both

def inspect(family, subclass=None, idx=0, biggest=False):
    matches = classify(family)
    if subclass is not None:
        matches = matches[matches['subclass'] == subclass]
    if biggest:
        matches = matches.sort_values('n_ids', ascending=False)
    if idx >= len(matches):
        print(f'{family}/{subclass}: only {len(matches)} groups available')
        return None
    return side_by_side(family, matches.iloc[idx])


#### A1.a `hard_dup`: every column agrees. Dropping either row loses no information.

In [9]:
inspect('bank_fixed_incomes', 'hard_dup')

[bank_fixed_incomes] subclass=hard_dup  snapshot=00c154d9  account=58365267  n_ids=2
  key: isin_code=BRBANKCDB0A0  issuer_cnpj=30680829000143  issue_date=2025-03-19 00:00:00  due_date=2028-03-01 00:00:00



,693a8d5b,69645925,differs
detail.snapshot_id,00c154d9-a32a-57bc-b2e2-45fb949909f3,00c154d9-a32a-57bc-b2e2-45fb949909f3,False
detail.snapshot_created_at,2026-08-13 09:55:05,2026-08-13 09:55:05,False
detail.institution_id,00000000000001,00000000000001,False
detail.institution_name,Nubank,Nubank,False
detail.party_id,41f11258-0e72-57e3-b1aa-d8ebdc720726,41f11258-0e72-57e3-b1aa-d8ebdc720726,False
detail.account_id,58365267-023f-5f84-a241-bae631b302a8,58365267-023f-5f84-a241-bae631b302a8,False
detail.connection_id,62f84bfa-8f8c-5593-b816-f7aae7db7bef,62f84bfa-8f8c-5593-b816-f7aae7db7bef,False
detail.ingested_at,2026-08-13 10:04:05,2026-08-13 10:04:05,False
detail.issuer_cnpj,30680829000143,30680829000143,False
detail.isin_code,BRBANKCDB0A0,BRBANKCDB0A0,False


#### A1.b `partition`: identity columns agree, quantities differ, gross amounts differ in proportion. These are real lots, so consumers must **sum** them, never pick one.

In [10]:
inspect('variable_incomes', 'partition', biggest=True)

[variable_incomes] subclass=partition  snapshot=1325f123  account=36c13618  n_ids=5
  key: ticker=B3SA3



,ae1bcec5,09d9db12,9fe25af5,a27af0b0,970ab59c,differs
detail.snapshot_id,1325f123-ecdf-5a34-88d9-f389cca9c98f,1325f123-ecdf-5a34-88d9-f389cca9c98f,1325f123-ecdf-5a34-88d9-f389cca9c98f,1325f123-ecdf-5a34-88d9-f389cca9c98f,1325f123-ecdf-5a34-88d9-f389cca9c98f,False
detail.snapshot_created_at,2026-08-15 09:54:56,2026-08-15 09:54:56,2026-08-15 09:54:56,2026-08-15 09:54:56,2026-08-15 09:54:56,False
detail.institution_id,00000000000006,00000000000006,00000000000006,00000000000006,00000000000006,False
detail.institution_name,C6 Bank,C6 Bank,C6 Bank,C6 Bank,C6 Bank,False
detail.party_id,e1c3e7ab-fc09-5174-a880-4319d1665fa6,e1c3e7ab-fc09-5174-a880-4319d1665fa6,e1c3e7ab-fc09-5174-a880-4319d1665fa6,e1c3e7ab-fc09-5174-a880-4319d1665fa6,e1c3e7ab-fc09-5174-a880-4319d1665fa6,False
detail.account_id,36c13618-4ea4-5208-bf90-8b9ce29d3b32,36c13618-4ea4-5208-bf90-8b9ce29d3b32,36c13618-4ea4-5208-bf90-8b9ce29d3b32,36c13618-4ea4-5208-bf90-8b9ce29d3b32,36c13618-4ea4-5208-bf90-8b9ce29d3b32,False
detail.connection_id,1dc10549-1ae9-509a-bdc0-1d2d3425e882,1dc10549-1ae9-509a-bdc0-1d2d3425e882,1dc10549-1ae9-509a-bdc0-1d2d3425e882,1dc10549-1ae9-509a-bdc0-1d2d3425e882,1dc10549-1ae9-509a-bdc0-1d2d3425e882,False
detail.ingested_at,2026-08-15 10:00:56,2026-08-15 10:00:56,2026-08-15 10:00:56,2026-08-15 10:00:56,2026-08-15 10:00:56,False
detail.issuer_cnpj,09346601000125,09346601000125,09346601000125,09346601000125,09346601000125,False
detail.isin_code,BRB3SAACNOR6,BRB3SAACNOR6,BRB3SAACNOR6,,BRB3SAACNOR6,True


#### A1.c `conflict`: identity and quantity agree, but gross amounts disagree. Typically one side is 0 and the other carries the real value. §3 decides what to do.

In [11]:
inspect('bank_fixed_incomes', 'conflict')

[bank_fixed_incomes] subclass=conflict  snapshot=04a468fc  account=454f5090  n_ids=2
  key: isin_code=BRBANKRDB1A1  issuer_cnpj=18236120000158  issue_date=2022-09-11 00:00:00  due_date=2025-09-01 00:00:00



,af6c3830,d00b2509,differs
detail.snapshot_id,04a468fc-82df-53eb-9ec2-42e6353c37ef,04a468fc-82df-53eb-9ec2-42e6353c37ef,False
detail.snapshot_created_at,2026-08-09 10:19:50,2026-08-09 10:19:50,False
detail.institution_id,00000000000003,00000000000003,False
detail.institution_name,Itau,Itau,False
detail.party_id,c7f2523c-161a-537c-80ad-77b16dccad08,c7f2523c-161a-537c-80ad-77b16dccad08,False
detail.account_id,454f5090-dddd-5e30-8fa6-2170b7ebe275,454f5090-dddd-5e30-8fa6-2170b7ebe275,False
detail.connection_id,a96839ae-d1c6-5a23-af33-e3b5468f7c6f,a96839ae-d1c6-5a23-af33-e3b5468f7c6f,False
detail.ingested_at,2026-08-09 10:22:50,2026-08-09 10:22:50,False
detail.issuer_cnpj,18236120000158,18236120000158,False
detail.isin_code,BRBANKRDB1A1,BRBANKRDB1A1,False


## 3 · Resolving the `conflict` case from §2

§2 left one case unresolved: same holding, same quantity, two different `gross` values. Which one do we keep?

The safe default would be to quarantine the whole group and let a human decide. But it turns out every conflict group in the data has the same, very specific shape, and that shape resolves the case on its own.

### The shape of every conflict group

- Exactly **two** `investment_id`s in the group.
- Both report the same **positive** quantity.
- One reports `gross = 0`, the other reports `gross > 0`.

### Why the zero side loses

A row saying "the account holds N shares worth zero in total" contradicts its sibling that prices the same N shares at a real number. Zero here is not a different opinion, it is a stale reading. Two pieces of evidence back this up:

1. **It is not a valid valuation.** `quantity > 0` and `gross = 0` means each share is worth zero, while the sibling prices them at a real number. One of the two has to be wrong, and it can only be the zero.
2. **The zero side was already zero last sync.** When we look at the previous sync's gross for that same `investment_id` (via `lag(gross)`), the zero side has been stuck at zero. The other id is the live one carrying the real value.

So the rule is: **keep the non-zero row, drop the zero row.** §3.3 states it formally.

### What the cell below actually does

For every row in every conflict group, it fetches the previous sync's gross for that same `investment_id` and then breaks the rows down two ways:

- **side**: `zero` if this row has `gross = 0`, `nonzero` if it has `gross > 0`.
- **history**: `first_appearance` if there was no previous sync; `frozen` if the previous gross equals the current gross; `moving` if it differed.

Every `zero` row should read as `frozen`: that is the defect signature (the zero side has been reporting zero for a while, hence the fossil metaphor).

The three asserts below then pin the shape: exactly two ids per group, exactly one at zero, and a shared positive quantity. If a future rebuild breaks any of them, the resolution rule no longer applies and the assert stops the notebook. That group would go to quarantine instead of being resolved automatically.

*(One quirk: a few `nonzero` rows may also show as `frozen`. That is just same-day double-syncs where the gross legitimately did not move between two calls. `frozen` on its own is not evidence of a defect; the case is the combination zero + positive quantity + a live sibling.)*


In [12]:
def conflict_members(family):
    cfg = FAMILIES[family]
    conflicts = classify(family)
    conflicts = conflicts[conflicts['subclass'] == 'conflict'].reset_index(drop=True)
    if conflicts.empty:
        return conflicts
    members = (conflicts.assign(group_id=conflicts.index)
               [['group_id', 'snapshot_id', 'account_id'] + cfg['key'] + ['investment_ids']]
               .explode('investment_ids').rename(columns={'investment_ids': 'investment_id'}))
    balance_history = query(f'''
        SELECT b.snapshot_id, b.investment_id, b.{cfg['qty']} AS qty, b.{cfg['gross']} AS gross,
               lag(b.{cfg['gross']}) OVER (
                   PARTITION BY b.investment_id
                   ORDER BY d.snapshot_created_at, b.snapshot_id) AS prev_gross
        FROM {balances(family)} b
        JOIN {detail(family)} d USING (snapshot_id, investment_id)''')
    return members.merge(balance_history, on=['snapshot_id', 'investment_id'], how='left')

members_by_family = {family: conflict_members(family) for family in FAMILIES}
conflict_rows = pd.concat(
    [members.assign(family=family) for family, members in members_by_family.items() if not members.empty],
    ignore_index=True)

conflict_rows['side'] = conflict_rows['gross'].eq(0).map({True: 'zero', False: 'nonzero'})
conflict_rows['history'] = conflict_rows.apply(
    lambda r: 'first_appearance' if pd.isna(r['prev_gross'])
    else ('frozen' if r['gross'] == r['prev_gross'] else 'moving'), axis=1)
print(pd.crosstab([conflict_rows['family'], conflict_rows['side']], conflict_rows['history']))

pair_shape = conflict_rows.groupby(['family', 'group_id']).agg(
    n_members=('investment_id', 'size'),
    n_zero=('gross', lambda s: int(s.eq(0).sum())),
    qty_min=('qty', 'min'))
assert pair_shape['n_members'].eq(2).all(), 'a conflict group with >2 ids exists: pair-based resolution below needs revisiting'
assert pair_shape['n_zero'].eq(1).all(), 'a conflict group without exactly one zero side exists: quarantine it, do not resolve'
assert pair_shape['qty_min'].gt(0).all(), 'a conflict group with zero/NULL quantity exists: the impossible-valuation argument does not apply'
print(f'\nall {len(pair_shape)} conflict groups are pairs: same positive quantity, one side gross=0, one side gross>0')
FINDINGS['conflict_both_nonzero'] = 0  # asserted above; if the assert fires, this becomes the quarantine count


history                     first_appearance  frozen  moving
family             side                                     
bank_fixed_incomes nonzero                 8       4      71
                   zero                    8      75       0

all 83 conflict groups are pairs: same positive quantity, one side gross=0, one side gross>0


### 3.1 · Transactions as a second witness

So far we have used only the position tables. But the transaction tables tell us how each position was reached, which is a second, independent source of evidence. We ask two questions of every conflict pair:

- **H1: do the two ids share history?** If they are really one holding, at a minimum they should share the origin purchase.
- **H2: does the live id carry the more recent movements?** If yes, transaction recency alone could tell us which id to keep and which to drop.

(An earlier version of this notebook tried a different transaction-based test: reconcile the running sum of `ENTRADA − SAIDA` movements against the delivered position quantity. Accrual events and portfolio rotation made the answer "wide gap" for almost every case, so the test was too blurry to be useful. We dropped it for the two sharper questions above.)

First, look at one pair's full movement stream. `carried_by` shows which id(s) each movement was booked under, and `n_deliveries` shows how many times the same movement was re-delivered across syncs:


In [13]:
def tx_stream(family, ids):
    cfg = FAMILIES[family]
    ids_sql = ', '.join(f"'{i}'" for i in ids)
    return query(f'''
        SELECT {cfg['tx_date']}::date AS dt, movement_type, transaction_type,
               {cfg['tx_qty']} AS qty, {cfg['tx_amount']} AS amount,
               string_agg(DISTINCT substr(investment_id, 1, 8), ' + ') AS carried_by,
               count(*) AS n_deliveries
        FROM {transactions(family)}
        WHERE investment_id IN ({ids_sql})
        GROUP BY ALL
        ORDER BY dt''')

first_pair = members_by_family['bank_fixed_incomes'].loc[lambda df: df['group_id'] == 0, 'investment_id'].tolist()
tx_stream('bank_fixed_incomes', first_pair)


,dt,movement_type,transaction_type,qty,amount,carried_by,n_deliveries
0,2022-09-11,ENTRADA,APLICACAO,115.114000,115114.0000,d00b2509 + af6c3830,26
1,2025-11-15,SAIDA,RESGATE,27.742812,27963.3674,af6c3830,13
2,2026-02-12,SAIDA,RESGATE,48.083172,47821.5996,af6c3830,13
3,2026-02-15,SAIDA,RESGATE,16.532490,16406.8427,af6c3830,13
4,2026-05-13,ENTRADA,APLICACAO,7.802954,7865.8461,af6c3830,13


Now test both questions across **every** distinct conflict pair. Note that the 83 conflict groups from §2 collapse down to a handful of physical pairs: the same pair of ids re-conflicts sync after sync, once per sync it appears in. Per pair we ask: do the two streams share their origin event, and whose stream is more recent?


In [14]:
def conflict_pair_tx(family):
    cfg = FAMILIES[family]
    members = members_by_family[family]
    per_id = (members.groupby('investment_id')
               .agg(times_zero=('gross', lambda s: int(s.eq(0).sum())),
                    times_nonzero=('gross', lambda s: int(s.ne(0).sum())))
               .reset_index())
    mixed_role = per_id[(per_id['times_zero'] > 0) & (per_id['times_nonzero'] > 0)]
    if len(mixed_role):
        print(f'note: {len(mixed_role)} id(s) sit on both sides across syncs; role assigned by majority')
    per_id['is_fossil'] = per_id['times_zero'] > per_id['times_nonzero']
    types_sql = ', '.join(f"'{mt}'" for mt in sorted(cfg['moving_tx']))
    tx_span = query(f'''
        SELECT investment_id,
               min({cfg['tx_date']})::date AS first_tx,
               max({cfg['tx_date']})::date AS last_tx,
               count(DISTINCT ({cfg['tx_date']}, transaction_type, movement_type, {cfg['tx_qty']})) AS n_events
        FROM {transactions(family)}
        WHERE transaction_type IN ({types_sql})
        GROUP BY 1''')
    per_id = per_id.merge(tx_span, on='investment_id', how='left')
    verdicts = []
    for pair_ids in members.groupby('group_id')['investment_id'].apply(lambda ids: tuple(sorted(ids))).drop_duplicates():
        pair_rows = per_id[per_id['investment_id'].isin(pair_ids)]
        fossil = pair_rows[pair_rows['is_fossil']].iloc[0]
        live = pair_rows[~pair_rows['is_fossil']].iloc[0]
        key_row = members[members['investment_id'] == pair_ids[0]].iloc[0]
        verdicts.append({
            'key': key_row[cfg['key'][0]],
            'fossil_id': fossil['investment_id'][:8], 'live_id': live['investment_id'][:8],
            'shared_origin': fossil['first_tx'] == live['first_tx'],
            'fossil_last_tx': fossil['last_tx'], 'live_last_tx': live['last_tx'],
            'fossil_more_recent': fossil['last_tx'] > live['last_tx'],
        })
    return pd.DataFrame(verdicts)

pair_verdicts = conflict_pair_tx('bank_fixed_incomes')
print(pair_verdicts.to_string(index=False))
print(f"\nH1 shared origin:  {int(pair_verdicts['shared_origin'].sum())}/{len(pair_verdicts)} pairs "
      f'-> CONFIRMED: transactions prove the two ids are one physical holding')
print(f"H2 recency picks survivor: fossil is the *more* recent side in {int(pair_verdicts['fossil_more_recent'].sum())}/{len(pair_verdicts)} pairs "
      f'-> REFUTED: the provider often books later movements on the dead id')


         key fossil_id  live_id  shared_origin fossil_last_tx live_last_tx  fossil_more_recent
BRBANKRDB1A1  af6c3830 d00b2509           True     2026-05-13   2022-09-11                True
BRBANKLCI2A2  d39f5068 f5e4d8a5           True     2026-08-11   2026-06-21                True
BRBANKCDB4A4  a8ffc513 ce1af572           True     2026-08-10   2026-07-22                True
BRBANKRDB5A5  56ed63a3 c72d0793           True     2026-08-15   2024-08-06                True
BRBANKCDB4A4  09690069 12923430           True     2019-11-10   2026-02-14               False
BRBANKCDB4A4  f5f26742 4ccb34ab           True     2026-08-07   2020-06-11                True
BRBANKLCI2A2  0021fddb 93e3e639           True     2026-01-26   2022-02-09                True
BRBANKLCA3A3  85537962 56836688           True     2026-08-12   2026-05-08                True

H1 shared origin:  8/8 pairs -> CONFIRMED: transactions prove the two ids are one physical holding
H2 recency picks survivor: fossil is the *mor

### 3.2 · What the two witnesses say

- **H1 confirmed.** Every conflict pair shares its origin transaction. That is independent proof, from a table we did not use to build the key, that the two ids really are one physical holding. The natural key and the transaction history point to the same conclusion.
- **H2 refuted.** In most pairs, the *fossil* (the zero side) is actually the one carrying the newer transactions. A dedup rule that said "keep the id with the most recent movements" would keep the wrong one. Worth knowing before anyone proposes it in a design review.

So we cannot use transaction recency to pick the survivor. Survivorship has to rest on the positions evidence alone, and that evidence is structural: every conflict group has the same positive quantity on both sides and exactly one side at zero (the asserts in §3 pin this).

### 3.3 · The resolution rule

| when we see... | decision | why |
|---|---|---|
| quantity agrees and is > 0, exactly one side has `gross = 0` | **admit** the live row with `dq_flags = ['zero_conflict_resolved']`, **reject** the zero row | zero is not a real valuation of a position the sibling prices. Quarantining both would silently drop a real holding from the portfolio. |
| quantity agrees, both sides have `gross > 0` but they disagree | **quarantine** the whole group | two live valuations, no tiebreaker: we cannot pick one, and summing would double-count. There are zero such groups today; the rule is kept as a guard for when it happens. |

Rejecting is safe because we never delete anything: the fossil row stays queryable in staging. It is only barred from the `holdings` view that a consumer sums up.


In [15]:
admitted = conflict_rows[conflict_rows['side'] == 'nonzero']
rejected = conflict_rows[conflict_rows['side'] == 'zero']
print(f"admit with warning : {len(admitted)} rows  (dq_flags = ['zero_conflict_resolved'])")
print(f'reject             : {len(rejected)} rows  (frozen-zero fossils; still queryable in staging)')
print(f"quarantine         : {FINDINGS['conflict_both_nonzero']} groups (both sides nonzero: none today, rule kept as guard)")


admit with warning : 83 rows  (dq_flags = ['zero_conflict_resolved'])
reject             : 83 rows  (frozen-zero fossils; still queryable in staging)
quarantine         : 0 groups (both sides nonzero: none today, rule kept as guard)


## 4 · A2: the same holding under a new id across syncs

The brief describes A2 as: *"the same holding reappearing later under a new identifier, so the old one freezes and the portfolio double-counts."* In this dataset, that pattern actually shows up in two different shapes, and we have already met both:

1. **Persistent fossil.** The old id keeps being delivered, frozen at zero, at the same time as its replacement. Because both ids sit under the same natural key, this is caught by the in-sync detector from §2, which reports it as a `conflict` (§3). No cross-sync logic needed.
2. **Clean handoff.** The old id simply stops being delivered, and a new id continues the key from the next sync onward. No sync ever contains both ids, so there is no double-count. The only visible effect is a discontinuity for anyone tracking `investment_id` directly (a position "disappears" and an unrelated one "appears").

**Detecting shape 2.** For every `(account, natural_key)` that has more than one id in its history, we look for:

- a **retired** id (its last-seen date is before the key's last-seen date), and
- a **late-born** id (its first-seen date is after the key's first-seen date).

If the late-born id's first sync is *at or before* the retired id's last sync, the two coexisted in some sync, which is a cross-sync double-count window. That window is also, by construction, an A1 duplicate group, so the assert cross-checks §2 and §4 against each other: if any overlap is found here, it should already appear as a duplicate group above.


In [16]:
def id_spans(family):
    key_cols = ', '.join(FAMILIES[family]['key'])
    return query(f'''
        SELECT account_id, {key_cols}, investment_id,
               min(snapshot_created_at) AS first_seen,
               max(snapshot_created_at) AS last_seen
        FROM {detail(family)}
        WHERE {key_ok(family, alias='')}
        GROUP BY ALL''')

def handoffs(family):
    spans = id_spans(family)
    key_cols = ['account_id'] + FAMILIES[family]['key']
    by_key = spans.groupby(key_cols, dropna=False)
    spans['key_first'] = by_key['first_seen'].transform('min')
    spans['key_last'] = by_key['last_seen'].transform('max')
    spans['n_ids'] = by_key['investment_id'].transform('nunique')
    spans['retired'] = spans['last_seen'] < spans['key_last']
    spans['lateborn'] = spans['first_seen'] > spans['key_first']
    multi_id = spans[spans['n_ids'] > 1]
    last_retired = multi_id[multi_id['retired']].groupby(key_cols, dropna=False)['last_seen'].max().rename('retired_last')
    first_lateborn = multi_id[multi_id['lateborn']].groupby(key_cols, dropna=False)['first_seen'].min().rename('lateborn_first')
    handoff = pd.concat([last_retired, first_lateborn], axis=1, join='inner').reset_index()
    handoff['overlaps'] = handoff['lateborn_first'] <= handoff['retired_last']
    return handoff

summary_rows = []
for family in FAMILIES:
    found = handoffs(family)
    summary_rows.append({'family': family, 'handoff_keys': len(found),
                         'overlapping': int(found['overlaps'].sum()) if len(found) else 0})
handoff_summary = pd.DataFrame(summary_rows)
print(handoff_summary.to_string(index=False))
FINDINGS['handoff_keys'] = int(handoff_summary['handoff_keys'].sum())

assert handoff_summary['overlapping'].sum() == 0, \
    'an old id and its replacement coexisted across syncs: check that the window shows up as an A1 duplicate group above'
print('\nevery handoff is clean: the old id stops before the new one starts: no cross-sync double-count in this dataset')


              family  handoff_keys  overlapping
    variable_incomes             0            0
               funds             0            0
  bank_fixed_incomes           120            0
credit_fixed_incomes            12            0
     treasure_titles             0            0

every handoff is clean: the old id stops before the new one starts: no cross-sync double-count in this dataset


### 4.1 · Watch one handoff happen

The timeline below shows one `(account, natural_key)` across every sync it appears in. Watch the id column change while the quantity column flows straight through unchanged. That is the whole point of the natural key: it stitches together what `investment_id` breaks.

- A consumer keyed on `investment_id` sees one position vanish and an unrelated one appear.
- A consumer keyed on the natural key sees one holding continuing without interruption.

So the decision for a clean handoff is to **admit**: nothing is wrong at holding grain. But we still attach `dq_flags = ['id_handoff']` on the first sync after the swap, because the discontinuity in `investment_id` is real metadata. Anything else that keys on the provider id (transaction history, cost-basis links) has a seam at that point, and downstream consumers should know.


In [17]:
bank_handoffs = handoffs('bank_fixed_incomes')
print(key_row := bank_handoffs.iloc[0][['account_id'] + FAMILIES['bank_fixed_incomes']['key']].to_string(), '\n')

def key_timeline(family, key_row):
    cfg = FAMILIES[family]
    filters = [f"d.account_id = '{key_row['account_id']}'"] + [f"d.{col} = '{key_row[col]}'" for col in cfg['key']]
    return query(f'''
        SELECT d.snapshot_created_at::date AS sync, substr(d.investment_id, 1, 8) AS id,
               b.{cfg['qty']} AS qty, b.{cfg['gross']} AS gross
        FROM {detail(family)} d
        JOIN {balances(family)} b USING (snapshot_id, investment_id)
        WHERE {' AND '.join(filters)}
        ORDER BY d.snapshot_created_at, d.snapshot_id''')

print(key_timeline('bank_fixed_incomes', bank_handoffs.iloc[0]).to_string(index=False))


account_id     02406856-4980-5215-9f87-76d6e9e08967
isin_code                              BRBANKLCI2A2
issuer_cnpj                          30306294000145
issue_date                      2021-12-31 00:00:00
due_date                        2024-12-01 00:00:00 

      sync       id   qty    gross
2026-07-29 24543054 24.67 24572.55
2026-07-31 24543054 24.67 24552.82
2026-08-02 24543054 24.67 24430.21
2026-08-04 24543054 24.67 24460.06
2026-08-08 24543054 24.67 24751.41
2026-08-10 75311440 24.67 24888.33
2026-08-13 75311440 24.67 24567.87
2026-08-14 75311440 24.67 24424.29
2026-08-17 75311440 24.67 24492.87
2026-08-20 75311440 24.67 24558.24
2026-08-21 75311440 24.67 24549.86


## 5 · A3: zero in one sync, back above zero in the next

The brief: *"a holding valued at zero in one sync and back above zero in the next with its quantity unchanged."*

A naive `gross = 0` filter would catch far too much: a position that closed and stays closed also reads as zero, but that is normal, not a defect. A real flap has a very specific shape:

> **previous sync gross > 0, this sync gross = 0, next sync gross > 0, quantity unchanged throughout**.

**Grain matters.** If we check this per `investment_id`, a frozen-zero fossil from §3 would show up here too and pollute the count. So we check it at **holding grain**: after grouping by natural key and summing all the lots that share it. That is also the grain a consumer actually reads. Note the split in the output below: fixed-income "zeros" are all sustained closures (positions that matured and stayed closed, not defects), while the genuine flaps are concentrated in `variable_incomes`, where a live position's price feed hiccups for one sync.

**Decision.** Admit the row with `dq_flags = ['zero_flap']`, and **keep the delivered zero as-is**. Never smooth it or interpolate it. The pipeline's job is to say "this zero is a lie, with quantity unchanged around it"; if we overwrote the delivered data, the pipeline would be the liar instead. This is the same reason §2's `partition_with_zero_lot` gets its own flag: a zero-priced lot inside a summed holding understates the total in the same way, just below the surface.


In [18]:
def holding_flaps(family):
    cfg = FAMILIES[family]
    key_cols = ', '.join(cfg['key'])
    return query(f'''
        WITH holding AS (
            SELECT d.account_id, {key_cols}, d.snapshot_id,
                   any_value(d.snapshot_created_at) AS synced_at,
                   sum(b.{cfg['gross']}) AS gross, sum(b.{cfg['qty']}) AS qty
            FROM {detail(family)} d
            JOIN {balances(family)} b USING (snapshot_id, investment_id)
            WHERE {key_ok(family)}
            GROUP BY d.account_id, {key_cols}, d.snapshot_id),
        w AS (
            SELECT *,
                   lag(gross)  OVER win AS prev_gross,
                   lead(gross) OVER win AS next_gross,
                   lag(qty)    OVER win AS prev_qty,
                   lead(qty)   OVER win AS next_qty
            FROM holding
            WINDOW win AS (PARTITION BY account_id, {key_cols} ORDER BY synced_at, snapshot_id))
        SELECT * FROM w
        WHERE gross = 0 AND prev_gross > 0 AND next_gross > 0
          AND qty = prev_qty AND qty = next_qty''')

flaps = {family: holding_flaps(family) for family in FAMILIES}
print(pd.DataFrame([{'family': family, 'holding_grain_flaps': len(found)} for family, found in flaps.items()]).to_string(index=False))
FINDINGS['zero_flaps'] = sum(len(found) for found in flaps.values())

for family, found in flaps.items():
    if len(found):
        cfg = FAMILIES[family]
        sample = found[['account_id'] + cfg['key'] + ['synced_at', 'prev_gross', 'gross', 'next_gross', 'qty']].copy()
        sample['account_id'] = sample['account_id'].str[:8]
        print(f'\n{family}: every flap, previous and next sync around the zero:')
        print(sample.to_string(index=False))


              family  holding_grain_flaps
    variable_incomes                   25
               funds                    0
  bank_fixed_incomes                    0
credit_fixed_incomes                    0
     treasure_titles                    0

variable_incomes: every flap, previous and next sync around the zero:
account_id ticker           synced_at  prev_gross  gross  next_gross     qty
  27859453 BPAC11 2026-08-06 10:04:11    75199.74    0.0    76177.93  1589.0
  77af5210 BPAC11 2026-08-13 10:15:36    96030.71    0.0    91019.62  1989.0
  495fc091 BPAC11 2026-08-11 10:29:49      616.91    0.0      634.77    13.0
  8d794832  PETR4 2026-08-11 10:02:02   208452.92    0.0   202950.54 20170.0
  0c5b7b36  PETR4 2026-08-11 10:23:18   109195.64    0.0   106554.88 10700.0
  0cf3dae5  VALE3 2026-08-08 10:24:21   141916.63    0.0   139334.03  8217.0
  bf884b6c  BBDC4 2026-08-16 10:10:43     3847.12    0.0     3771.92   120.0
  e723c3c9  B3SA3 2026-08-07 09:58:31   110051.49    0.0   11

## 6 · The decision matrix

This table is built from the live counts each section wrote into `FINDINGS`, so if the data changes, this table changes with it.

The three verbs from the brief map onto the pipeline as follows:

- **reject** = the row does not appear in the `holdings` view, but it stays in the staging table. Nothing is ever deleted.
- **quarantine** = the whole group is held back into its own relation (`holdings_quarantine`). It is missing from `holdings`, but its absence is always explainable by its presence somewhere else.
- **admit with warning** = the row is in the `holdings` view, and `dq_flags` tells the consumer why to look twice.

A consumer finds out what happened the same way in every case: check `dq_flags` on the holding, and check `holdings_quarantine` for anything you expected but do not see. A holding with `dq_flags = []` is fully clean.


In [19]:
matrix = pd.DataFrame([
    {'defect': 'A1.a hard duplicate: same key, qty and gross agree', 'occurrences': FINDINGS['hard_dup'],
     'decision': 'reject the redundant copies, keep one',
     'consumer sees': 'nothing - dedup is lossless'},
    {'defect': 'A1.b partition: same key, quantities differ (real lots)', 'occurrences': FINDINGS['partition'],
     'decision': 'admit every lot, SUM at holding grain',
     'consumer sees': "n_lots > 1, dq_flags ['merged_lots']"},
    {'defect': 'A1.b partition containing a zero-gross lot', 'occurrences': FINDINGS['partition_with_zero_lot'],
     'decision': 'admit and sum, flag that the total may understate',
     'consumer sees': "dq_flags ['zero_gross_lot']"},
    {'defect': 'A1.c conflict: same key+qty, one side gross=0 (fossil)', 'occurrences': FINDINGS['conflict'],
     'decision': 'admit the live row, reject the frozen-zero fossil',
     'consumer sees': "dq_flags ['zero_conflict_resolved']"},
    {'defect': 'A1.c conflict: same key+qty, both sides gross>0', 'occurrences': FINDINGS['conflict_both_nonzero'],
     'decision': 'quarantine the whole group',
     'consumer sees': 'holding absent + row in quarantine relation'},
    {'defect': 'A2 handoff: id retires, new id continues the key', 'occurrences': FINDINGS['handoff_keys'],
     'decision': 'admit - the natural key stitches identity',
     'consumer sees': "dq_flags ['id_handoff'] on the sync after the swap"},
    {'defect': 'A3 zero-flap: holding gross 0 then >0, qty unchanged', 'occurrences': FINDINGS['zero_flaps'],
     'decision': 'admit with warning, keep the delivered zero',
     'consumer sees': "dq_flags ['zero_flap']"},
    {'defect': 'blank key column: row cannot be identified at all (§1.2)', 'occurrences': FINDINGS['missing_identity_rows'],
     'decision': 'admit at lot grain, never merged with anything',
     'consumer sees': "dq_flags ['missing_identity']"},
])
matrix


,defect,occurrences,decision,consumer sees
0,"A1.a hard duplicate: same key, qty and gross agree",1428,"reject the redundant copies, keep one",nothing - dedup is lossless
1,"A1.b partition: same key, quantities differ (real lots)",2975,"admit every lot, SUM at holding grain","n_lots > 1, dq_flags ['merged_lots']"
2,A1.b partition containing a zero-gross lot,101,"admit and sum, flag that the total may understate",dq_flags ['zero_gross_lot']
3,"A1.c conflict: same key+qty, one side gross=0 (fossil)",83,"admit the live row, reject the frozen-zero fossil",dq_flags ['zero_conflict_resolved']
4,"A1.c conflict: same key+qty, both sides gross>0",0,quarantine the whole group,holding absent + row in quarantine relation
5,"A2 handoff: id retires, new id continues the key",132,admit - the natural key stitches identity,dq_flags ['id_handoff'] on the sync after the swap
6,"A3 zero-flap: holding gross 0 then >0, qty unchanged",25,"admit with warning, keep the delivered zero",dq_flags ['zero_flap']
7,blank key column: row cannot be identified at all (§1.2),224,"admit at lot grain, never merged with anything",dq_flags ['missing_identity']


## 7 · Bringing it to dbt

Here is how each piece maps onto the repo's canonical / consumption layer split.

**Detection lives at staging as dbt tests.** One dbt test per family, compiled from `classify_sql` in §2, severity `warn`, `store_failures: true`. The audit table `main_dbt_test__audit` then becomes the persistent version of the §2.1 census, and §2.2's attribution is one `GROUP BY` on it.

**Blank identity is normalized at staging.** The trap from §1.2 (`''` passes `IS NOT NULL`) is killed at the source: staging wraps every identity column in `nullif(trim(isin_code), '')`, so a blank string *becomes* a NULL. From then on, every `IS NOT NULL` check and every join behaves correctly. Rows whose key is NULL after that transformation skip dedup and merging entirely: they enter the holdings view at lot grain with `dq_flags = ['missing_identity']`. Merging rows we cannot identify is how double-counts are born, so we don't merge them.

**The intermediate layer (`int_*`) stays untouched.** Its grain is `(snapshot_id, investment_id)` (one row per delivered lot), which is correct input for every subclass here. All the resolution logic lives one layer above, in `consumption`, so nothing is destroyed before a human has a chance to look at it.

**Resolution is one `holdings_<family>` view in consumption**, at grain `(snapshot_id, account_id, natural_key)`. Every decision from the matrix is implemented as one CTE. Sketch below is for `bank_fixed_incomes`; the only thing that varies per family is the natural key, so this templates cleanly into a macro driven by the `FAMILIES` config.

```sql
with lots as (
    select * from {{ ref('int_bank_fixed_incomes_positions') }}
),

-- A1.a: reject byte-identical copies (same sync, key, qty, gross), keep the lowest id
dedup as (
    select * from lots
    qualify row_number() over (
        partition by snapshot_id, account_id,
                     isin_code, issuer_cnpj, issue_date, due_date,
                     quantity, gross_amount
        order by investment_id) = 1
),

-- A1.c: reject the frozen-zero fossil when a sibling prices the same quantity
conflicts_resolved as (
    select *,
        gross_amount = 0
            and quantity > 0
            and max(gross_amount) over (
                    partition by snapshot_id, account_id,
                                 isin_code, issuer_cnpj, issue_date, due_date, quantity
                ) > 0
            as is_zero_fossil
    from dedup
    qualify not is_zero_fossil
),

-- A1.b: what remains are real lots -> sum to holding grain
holdings as (
    select
        snapshot_id, account_id,
        isin_code, issuer_cnpj, issue_date, due_date,
        any_value(snapshot_created_at)                as synced_at,
        sum(quantity)                                 as quantity,
        sum(gross_amount)                             as gross_amount,
        count(*)                                      as n_lots,
        bool_or(is_zero_fossil)                       as resolved_zero_conflict,
        bool_or(gross_amount = 0 and quantity > 0)    as has_zero_gross_lot,
        list(investment_id order by investment_id)    as lot_ids
    from conflicts_resolved
    group by all
),

-- A2 + A3 need the holding's own history: one window pass
flagged as (
    select *,
        list_filter([
            case when n_lots > 1                 then 'merged_lots'            end,
            case when resolved_zero_conflict     then 'zero_conflict_resolved' end,
            case when has_zero_gross_lot         then 'zero_gross_lot'         end,
            case when lag(lot_ids)  over w is not null
                  and lot_ids != lag(lot_ids) over w
                                                 then 'id_handoff'             end,
            case when gross_amount = 0
                  and lag(gross_amount)  over w > 0
                  and lead(gross_amount) over w > 0
                  and quantity = lag(quantity) over w
                                                 then 'zero_flap'              end
        ], x -> x is not null) as dq_flags
    from holdings
    window w as (
        partition by account_id, isin_code, issuer_cnpj, issue_date, due_date
        order by synced_at, snapshot_id)
)

select * from flagged
```

A few notes on the sketch above:

- The `is_zero_fossil` flag inside `qualify` implements the resolution rule from §3.3 exactly. Conflict groups where both sides are nonzero fall through *unresolved* into a separate model (`holdings_quarantine`) that selects them by: same key and quantity, more than one distinct gross, and no zero side. Quarantine is a real relation, not a filter: a consumer can always explain a missing holding by looking there.
- The `zero_flap` flag uses `lead()`, so we can only assign it one sync in arrears (we need the *next* sync to see the recovery). In a view this recomputes for free every time; in an incremental model, the last sync would need to be reprocessed each run.
- `lot_ids` serves double duty: it drives the `id_handoff` detector (the flag fires when the list of contributing ids changes) and it doubles as lineage, letting a consumer trace any holding back to the exact provider rows it was built from.

**Tests in `schema.yml` that keep this honest:**

1. Uniqueness on `(snapshot_id, account_id, natural_key)` in the holdings view: the grain contract.
2. `error` if `holdings_quarantine` gains rows: today it is empty, and §3's asserts say it should stay empty. Growth means the provider invented a new failure mode and someone needs to look.
3. `warn` monitors on handoff and flap counts: these are provider-quality regressions, and §2.2's attribution routes them to whoever owns each integration.

**The consumer contract, in one paragraph.** Read `holdings_*`. If `dq_flags = []`, no across-record defect touched that number. If any flag is present, the value is still usable, but it comes with a documented caveat: one of `merged_lots`, `zero_conflict_resolved`, `zero_gross_lot`, `id_handoff`, `zero_flap`, or `missing_identity`. If you expected a holding and cannot find it, it is either in `holdings_quarantine` (the pipeline refused to guess between two live valuations) or it was never delivered by the provider. In either case, every raw row is still queryable in staging.
